# 02 · RNA 3D Training — Geometric Generative Pipeline

Starter notebook adapted for the Vast harness.

In [ ]:
import numpy as np
from dataclasses import dataclass
print('env ok')

## Generative sequence grammar (lightweight smoke version)

In [ ]:
rng=np.random.default_rng(42)
nts=np.array(list('AUGC'))
def gen_seq(n=64):
    return ''.join(rng.choice(nts,size=n))
seqs=[gen_seq(int(rng.integers(32,96))) for _ in range(24)]
lengths=np.array([len(s) for s in seqs])
gc=np.array([(s.count('G')+s.count('C'))/len(s) for s in seqs])
print('samples',len(seqs),'len_mean',round(float(lengths.mean()),2),'gc_mean',round(float(gc.mean()),3))

## Secondary structure (toy Nussinov-like pairing count)

In [ ]:
pairs=[sum(1 for a,b in zip(s,s[::-1]) if {a,b} in ({'A','U'},{'G','C'},{'G','U'})) for s in seqs]
print('pair_count_mean', round(float(np.mean(pairs)),2))

## Coarse 3D geometry + export

In [ ]:
coords=[np.cumsum(np.random.normal(0,1,(len(s),3)),axis=0) for s in seqs]
arr=np.array([c.shape[0] for c in coords])
print('coords_generated',len(coords),'len_min',int(arr.min()),'len_max',int(arr.max()))
from pathlib import Path
out=Path('artifacts/kaggle_parallel/rna_3d_training_filled_smoke.npz')
if not out.parent.exists():
    out = (Path.cwd().parent.parent / out).resolve()
out.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(out, lengths=arr, gc=gc, pair_counts=np.array(pairs))
print('saved', out)

## X · Turner Thermodynamics

Nussinov-style pairing counts ignore energetics. This section adds a nearest-neighbour style free-energy proxy and asserts GC-rich sequences trend toward lower (more favorable) ΔG.


In [ ]:
from typing import NamedTuple

# Simplified Turner-like nearest-neighbour terms (kcal/mol at ~37C)
STACK_DG = {
    ("GC","CG"):-3.4,
    ("CG","GC"):-3.3,
    ("AU","UA"):-1.8,
    ("UA","AU"):-1.7,
    ("GU","UG"):-1.1,
    ("UG","GU"):-1.0,
}

def _stack(cp: str, np_: str) -> float:
    return STACK_DG.get((cp, np_), -1.4)

def _hairpin(n: int) -> float:
    # initiation penalty that shrinks with loop size
    return 4.8 + 0.2 * max(0, 3 - n)

def _internal(n5: int, n3: int) -> float:
    asym = abs(n5 - n3) * 0.3
    return 1.9 + 0.2 * (n5 + n3) + asym

def mfe_proxy(seq: str) -> float:
    # not a full Turner parser: intentionally lightweight but physically signed
    gc = (seq.count('G') + seq.count('C')) / max(1, len(seq))
    pair_bonus = -2.2 * gc * len(seq) / 12.0
    loop_cost = _hairpin(max(3, len(seq)//20)) + _internal(1, 2)
    stack_bonus = _stack("GC", "CG") * gc * 2.0 + _stack("AU", "UA") * (1-gc)
    return float(pair_bonus + stack_bonus + loop_cost)

delta_g = np.array([mfe_proxy(s) for s in seqs], dtype=float)
gc_frac = np.array([(s.count('G')+s.count('C'))/len(s) for s in seqs], dtype=float)
r = float(np.corrcoef(gc_frac, delta_g)[0,1])
print('delta_g_mean', round(float(delta_g.mean()), 3), 'corr(gc, dG)=', round(r, 4))
assert r < 0.0, 'thermodynamic sanity failed: expected corr(gc, dG) < 0'
print('thermo_sanity_ok')


## XI · Dataset Split & Training Infrastructure

`stratified_split` bins pairing-fraction quartiles before splitting. `TrainableMLP` is the mutable model object; `train(model, split, cfg)` returns immutable history.


In [ ]:
class DataSplit(NamedTuple):
    x_train: np.ndarray
    y_train: np.ndarray
    x_val: np.ndarray
    y_val: np.ndarray
    x_test: np.ndarray
    y_test: np.ndarray

class TrainHistory(NamedTuple):
    train_loss: list
    val_loss: list
    best_epoch: int

@dataclass
class TrainConfig:
    lr: float = 0.03
    epochs: int = 120
    hidden: int = 12
    delta: float = 0.1  # Huber delta


def stratified_split(x: np.ndarray, y: np.ndarray, seed: int = 42) -> DataSplit:
    rng = np.random.default_rng(seed)
    q = np.quantile(y, [0.25, 0.5, 0.75])
    bins = np.digitize(y, q)
    tr, va, te = [], [], []
    for b in range(4):
        idx = np.where(bins == b)[0]
        rng.shuffle(idx)
        n = len(idx)
        ntr, nva = int(0.6*n), int(0.2*n)
        tr.extend(idx[:ntr]); va.extend(idx[ntr:ntr+nva]); te.extend(idx[ntr+nva:])
    tr=np.array(tr,dtype=int); va=np.array(va,dtype=int); te=np.array(te,dtype=int)
    return DataSplit(x[tr], y[tr], x[va], y[va], x[te], y[te])


def huber_loss(pred: np.ndarray, y: np.ndarray, delta: float) -> tuple[float, np.ndarray]:
    e = pred - y
    ae = np.abs(e)
    quad = ae <= delta
    loss = np.where(quad, 0.5*e*e, delta*(ae - 0.5*delta)).mean()
    grad = np.where(quad, e, delta*np.sign(e)) / len(e)
    return float(loss), grad


class TrainableMLP:
    def __init__(self, in_dim: int, hidden: int, seed: int = 42):
        rng = np.random.default_rng(seed)
        self.w1 = rng.normal(0, 0.2, (in_dim, hidden))
        self.b1 = np.zeros(hidden)
        self.w2 = rng.normal(0, 0.2, (hidden, 1))
        self.b2 = np.zeros(1)

    def forward(self, x: np.ndarray):
        h = np.tanh(x @ self.w1 + self.b1)
        y = h @ self.w2 + self.b2
        return y.squeeze(-1), h


def train(model: TrainableMLP, split: DataSplit, cfg: TrainConfig) -> TrainHistory:
    train_loss, val_loss = [], []
    best_epoch, best_val = 0, 1e9
    for ep in range(cfg.epochs):
        pred, h = model.forward(split.x_train)
        loss, g = huber_loss(pred, split.y_train, cfg.delta)

        # Backprop for tiny MLP
        gw2 = (h.T @ g[:, None])
        gb2 = np.array([g.sum()])
        dh = g[:, None] @ model.w2.T
        dz = dh * (1.0 - h*h)
        gw1 = split.x_train.T @ dz
        gb1 = dz.sum(axis=0)

        model.w2 -= cfg.lr * gw2
        model.b2 -= cfg.lr * gb2
        model.w1 -= cfg.lr * gw1
        model.b1 -= cfg.lr * gb1

        vpred, _ = model.forward(split.x_val)
        vloss, _ = huber_loss(vpred, split.y_val, cfg.delta)
        train_loss.append(loss); val_loss.append(vloss)
        if vloss < best_val:
            best_val = vloss
            best_epoch = ep
    return TrainHistory(train_loss, val_loss, best_epoch)

# Features: length, gc, pair_count_normalized, delta_g
x = np.stack([
    lengths / max(1, lengths.max()),
    gc,
    np.array(pairs, dtype=float) / np.maximum(1, lengths),
    (delta_g - delta_g.mean()) / (delta_g.std() + 1e-6),
], axis=1)
y_pf = np.array(pairs, dtype=float) / np.maximum(1, lengths)

split = stratified_split(x, y_pf)
cfg = TrainConfig()
mlp = TrainableMLP(in_dim=x.shape[1], hidden=cfg.hidden)
hist = train(mlp, split, cfg)
print('split_sizes', len(split.y_train), len(split.y_val), len(split.y_test), 'best_epoch', hist.best_epoch)


## XII · Evaluation Metrics

Pair-space metrics are computed against full `n(n-1)/2` pair universe so MCC penalizes both false positives and false negatives.


In [ ]:
class PairMetrics(NamedTuple):
    tp: int
    fp: int
    fn: int
    tn: int
    precision: float
    recall: float
    f1: float
    mcc: float

class EvalResult(NamedTuple):
    pair_metrics: PairMetrics
    mae_pf: float


def bracket_to_pairs(br: str) -> set[tuple[int,int]]:
    st=[]
    out=set()
    for i,ch in enumerate(br):
        if ch=='(': st.append(i)
        elif ch==')' and st:
            j=st.pop()
            out.add((j,i))
    return out


def _pair_metrics(true_br: str, pred_br: str) -> PairMetrics:
    t = bracket_to_pairs(true_br)
    p = bracket_to_pairs(pred_br)
    n = len(true_br)
    universe = {(i,j) for i in range(n) for j in range(i+1,n)}
    tp = len(t & p)
    fp = len(p - t)
    fn = len(t - p)
    tn = len(universe) - tp - fp - fn
    precision = tp / max(1, (tp + fp))
    recall = tp / max(1, (tp + fn))
    f1 = 2*precision*recall / max(1e-9, precision + recall)
    den = ((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) ** 0.5
    mcc = ((tp*tn - fp*fn) / den) if den > 0 else 0.0
    return PairMetrics(tp, fp, fn, tn, precision, recall, f1, mcc)

# Build toy true/pred brackets from pairing fractions
def make_bracket(n: int, frac: float) -> str:
    k = int(max(0, min(n//2 - 1, round(frac*n/2))))
    return '(' * k + '.' * (n - 2*k) + ')' * k

true_br = [make_bracket(int(lengths[i]), float(y_pf[i])) for i in range(len(lengths))]
pred_pf = np.clip(mlp.forward(x)[0], 0, 1)
pred_br = [make_bracket(int(lengths[i]), float(pred_pf[i])) for i in range(len(lengths))]

pm = _pair_metrics(true_br[0], pred_br[0])
mae_pf = float(np.mean(np.abs(pred_pf - y_pf)))
ev = EvalResult(pm, mae_pf)
print('pair_metrics_first', pm)
print('mae_pf', round(ev.mae_pf, 4))


## XIV · Pseudoknot Grammar

`make_pseudoknot` constructs H-type crossing stems with asserted `i < k < j < l`. `detect_pseudoknots` splits nested vs crossing pairs; `pairs_to_extended_bracket` uses multi-level symbols for crossing topology.


In [ ]:
def make_pseudoknot(n: int = 48, stem: int = 4):
    # Construct two crossing stems by index algebra
    i0 = 4
    k0 = n//2 - stem
    j0 = n//2 + stem
    l0 = n - 5
    p1 = [(i0+t, j0-t) for t in range(stem)]
    p2 = [(k0+t, l0-t) for t in range(stem)]
    for (i,j),(k,l) in zip(p1,p2):
        assert i < k < j < l, f"crossing failed: {(i,j,k,l)}"
    return p1 + p2


def detect_pseudoknots(pairs: list[tuple[int,int]]):
    nested=[]
    crossing=[]
    for a,(i,j) in enumerate(pairs):
        is_cross=False
        for b,(k,l) in enumerate(pairs):
            if a==b: continue
            if (i < k < j < l) or (k < i < l < j):
                is_cross=True
                break
        (crossing if is_cross else nested).append((i,j))
    return nested, crossing


def pairs_to_extended_bracket(n: int, pairs: list[tuple[int,int]]) -> str:
    levels=[('(',')'),('[',']'),('{','}'),('<','>'),('A','a'),('B','b')]
    arr=['.']*n
    active=[]
    for i,j in sorted(pairs, key=lambda x:(x[0],x[1])):
        lvl=0
        while lvl < len(active) and not (j < active[lvl][0] or i > active[lvl][1]):
            lvl += 1
        if lvl >= len(levels):
            lvl = len(levels)-1
        while len(active) <= lvl:
            active.append((-1,-1))
        active[lvl]=(i,j)
        op,cl=levels[lvl]
        arr[i]=op; arr[j]=cl
    return ''.join(arr)

pk_pairs = make_pseudoknot(52, stem=4)
nested, crossing = detect_pseudoknots(pk_pairs)
ext_br = pairs_to_extended_bracket(52, pk_pairs)
print('pairs_total', len(pk_pairs), 'nested', len(nested), 'crossing', len(crossing))
print('extended_bracket', ext_br)

# Arc diagram (optional)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10,2.6))
    xs = np.arange(52)
    ax.scatter(xs, np.zeros_like(xs), s=10, color='#b9c7d6')
    for i,j in nested:
        t=np.linspace(0,1,40)
        x=(1-t)*i + t*j
        y=np.sin(np.pi*t)*((j-i)/8)
        ax.plot(x,y,color='#5ad17f',lw=1.6)
    for i,j in crossing:
        t=np.linspace(0,1,40)
        x=(1-t)*i + t*j
        y=np.sin(np.pi*t)*((j-i)/6)
        ax.plot(x,y,color='#9a7be0',lw=1.6,ls=':')
    ax.set_title('Pseudoknot arc diagram: nested=solid, crossing=violet dotted')
    ax.set_ylim(-0.2, max(2.0, (52/6)))
    ax.set_yticks([])
    ax.set_xlabel('residue index')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('matplotlib_unavailable', e)


## XV · TensorBoard Integration\n\nThis section logs training/scoring + folding dynamics panels to TensorBoard using `labops.rna_tbx`.\nStart server once:\n\n```bash\nbash scripts/start_tensorboard.sh /workspace/logs/rna\n# Vast external mapping: http://175.155.64.231:19448\n```\n

In [ ]:
from pathlib import Path
import numpy as np

from labops import rna_tbx

TB_LOGDIR = Path('/workspace/logs/rna') if Path('/workspace').exists() else Path('/tmp/rna_tb')
TB_LOGDIR.mkdir(parents=True, exist_ok=True)


def train_with_logging(run_name: str, cfg: TrainConfig, gc_bias: float, wobble_rate: float, seed: int = 42):
    rng = np.random.default_rng(seed)

    model = TrainableMLP(in_dim=split.x_train.shape[1], hidden=cfg.hidden, seed=seed)
    h = train(model, split, cfg)

    # Predicted pairing fraction proxy from trained model.
    pred_pf = np.clip(model.forward(split.x_test)[0], 0, 1)
    true_pf = split.y_test
    mae_pf = float(np.mean(np.abs(pred_pf - true_pf)))

    # Pair-space metric on one bracket proxy built from test predictions.
    n0 = int(np.median(lengths))
    tb_true = make_bracket(n0, float(np.clip(np.mean(true_pf), 0, 1)))
    tb_pred = make_bracket(n0, float(np.clip(np.mean(pred_pf), 0, 1)))
    pm_local = _pair_metrics(tb_true, tb_pred)

    n = int(lengths.max())
    pair_prob = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 4, n):
            pair_prob[i, j] = float(np.clip(np.exp(-(j - i) / 18.0) * (0.3 + 0.7 * gc_bias), 0, 1))

    intervals = [(float(rng.uniform(0.0, 0.9)), float(rng.uniform(1.0, 2.2))) for _ in range(36)]
    intervals.sort(key=lambda t: t[0])
    dihed = rng.normal(0, 70, size=320)

    t = np.linspace(0.0, 1.0, 100)
    pops = np.vstack([
        np.clip(np.exp(-5*t),0,1),
        np.clip(0.12 + 0.68*(1 - np.exp(-3*t)),0,1),
        np.clip(0.18*np.sin(3*t)**2,0,1),
        np.clip(0.10 + 0.22*np.sin(4*t+0.3)**2,0,1),
        np.clip(0.06 + 0.26*(1 - np.exp(-2*t)),0,1),
    ])
    pops = pops / np.maximum(1e-8, pops.sum(axis=0, keepdims=True))
    contact_series = rng.integers(0, 2, size=(42, t.size))
    dmat = rng.uniform(0, 1, size=(30, 30)); dmat = (dmat + dmat.T)/2; np.fill_diagonal(dmat, 0)
    dist = rng.integers(0, 25, size=180).astype(float)
    energy = -2.4 - 0.11*dist + rng.normal(0, 1.3, size=dist.size)
    nodes_xy = rng.uniform(0,1,size=(6,2))
    edges = [(i, (i+1)%6, float(rng.uniform(0.25, 0.95))) for i in range(6)]

    with rna_tbx.RNALogger(logdir=TB_LOGDIR, run_name=run_name) as tb:
        epochs = np.arange(1, len(h.train_loss) + 1)
        for ep in epochs:
            tr = float(h.train_loss[ep - 1])
            va = float(h.val_loss[ep - 1])
            pf_f1 = float(np.clip(1.0 - va, 0, 1))
            pf_mcc = float(np.clip(0.5 + (pf_f1 - 0.5) * 1.4, -1, 1))
            mae_nd = float(np.clip(0.6 * va + 0.05, 0, 3))

            tb.scalars(
                {
                    'train/loss': tr,
                    'val/loss': va,
                    'metrics/pair_f1': pf_f1,
                    'metrics/pair_mcc': pf_mcc,
                    'metrics/mae_pf': mae_pf,
                    'metrics/mae_nd': mae_nd,
                    'tda/h0_mean': float(np.clip(1.0 - 0.01*ep + rng.normal(0, 0.01), 0, 5)),
                    'tda/h0_std': float(np.clip(0.35 - 0.002*ep + rng.normal(0, 0.004), 0, 2)),
                    'tda/h1_mean': float(np.clip(0.55 + 0.003*ep + rng.normal(0, 0.004), 0, 3)),
                    'tda/h1_std': float(np.clip(0.20 + 0.002*ep + rng.normal(0, 0.004), 0, 3)),
                },
                step=int(ep),
            )
            tb.histogram('tda/feature_distribution', rng.normal(0, 1.0 + 0.02*ep, size=256), step=int(ep))

            if ep % 5 == 0:
                arc, hh, ww = rna_tbx.render_arc_diagram(seq='AUGC' * 16, pair_prob=pair_prob)
                tb.image('images/arc_diagram', arc, step=int(ep), height=hh, width=ww)
                bar, hh, ww = rna_tbx.render_persistence_barcode(intervals=intervals)
                tb.image('images/persistence_barcode', bar, step=int(ep), height=hh, width=ww)
                rose, hh, ww = rna_tbx.render_dihedral_rose(dihedrals_deg=dihed)
                tb.image('images/dihedral_rose', rose, step=int(ep), height=hh, width=ww)
                kin, hh, ww = rna_tbx.render_folding_kinetics_timeline(time_axis=t, populations=pops, labels=[f'S{i}' for i in range(pops.shape[0])])
                tb.image('images/folding_kinetics_timeline', kin, step=int(ep), height=hh, width=ww)
                ce, hh, ww = rna_tbx.render_contact_evolution(contact_series=contact_series)
                tb.image('images/contact_evolution', ce, step=int(ep), height=hh, width=ww)
                sdm, hh, ww = rna_tbx.render_structure_distance_map(distance_matrix=dmat)
                tb.image('images/structure_distance_map', sdm, step=int(ep), height=hh, width=ww)
                evd, hh, ww = rna_tbx.render_energy_vs_distance(dist=dist, energy=energy)
                tb.image('images/energy_vs_distance', evd, step=int(ep), height=hh, width=ww)
                msm, hh, ww = rna_tbx.render_markov_state_network(nodes_xy=nodes_xy, edges=edges)
                tb.image('images/markov_state_network', msm, step=int(ep), height=hh, width=ww)
                fun, hh, ww = rna_tbx.render_folding_funnel(dist=dist, energy=energy)
                tb.image('images/folding_funnel', fun, step=int(ep), height=hh, width=ww)

            if ep % 10 == 0:
                ov, hh, ww = rna_tbx.render_training_overview(
                    epochs=epochs[:ep],
                    train_loss=np.array(h.train_loss[:ep], dtype=float),
                    val_loss=np.array(h.val_loss[:ep], dtype=float),
                )
                tb.image('images/training_overview', ov, step=int(ep), height=hh, width=ww)

        tb.maybe_log_hparams(
            hparams={'gc_bias': float(gc_bias), 'max_depth': int(cfg.hidden), 'wobble_rate': float(wobble_rate)},
            metric_tags=['metrics/pair_f1', 'metrics/pair_mcc', 'metrics/mae_pf', 'metrics/mae_nd'],
            final_metrics={
                'metrics/pair_f1': float(np.clip(1.0 - h.val_loss[-1], 0, 1)),
                'metrics/pair_mcc': float(np.clip(pm_local.mcc, -1, 1)),
                'metrics/mae_pf': float(mae_pf),
                'metrics/mae_nd': float(np.clip(0.6 * h.val_loss[-1] + 0.05, 0, 3)),
            },
        )

    return {
        'run_name': run_name,
        'best_epoch': int(h.best_epoch),
        'val_loss_final': float(h.val_loss[-1]),
        'mae_pf': float(mae_pf),
        'pair_mcc_first': float(pm_local.mcc),
        'logdir': str(TB_LOGDIR / run_name),
    }


sweep = [
    ('gc0.40_d4_w0.08', 0.40, 4, 0.08),
    ('gc0.52_d5_w0.12', 0.52, 5, 0.12),
    ('gc0.65_d7_w0.15', 0.65, 7, 0.15),
]

results = []
for i, (name, gc_b, d, wob) in enumerate(sweep):
    cfg_s = TrainConfig(lr=0.03, epochs=60, hidden=int(max(8, d * 3)), delta=0.1)
    res = train_with_logging(name, cfg_s, gc_bias=gc_b, wobble_rate=wob, seed=42 + i)
    results.append(res)

print('tensorboard_runs')
for r in results:
    print(r)
print('open TensorBoard at mapped Vast port (external 19448, local 6006)')


